## Example 1: jupyter_server functionality test 1

In [ ]:
import subprocess
import sys
import textwrap

from jupyter_server.serverapp import ServerApp

# Example 1: ServerApp configuration and initialization
print("=== Example 1: ServerApp Configuration ===")

# Create and configure a ServerApp (configuration only; nothing is started in this kernel)
app = ServerApp()
app.port = 0  # let the OS pick a free port
app.open_browser = False
print(f"Created ServerApp: {type(app).__name__}")
print(f"Configured port: {app.port}")
print(f"Browser opening disabled: {not app.open_browser}")
assert app.port == 0 and app.open_browser is False

# initialize() installs signal handlers and sets up the server's IOLoop, so run it in a subprocess
# so the server's signal handlers and IOLoop do not affect this kernel.
script = textwrap.dedent("""
    from jupyter_server.serverapp import ServerApp
    app = ServerApp()
    app.port = 0
    app.open_browser = False
    app.initialize([])
    print("initialized", type(app).__name__)
""")
done = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=120)
assert done.returncode == 0, f"ServerApp.initialize failed: {done.stderr[-400:]}"
assert "initialized ServerApp" in done.stdout, done.stdout
print("\u2713 ServerApp initialized successfully (in a subprocess)")

print(f"Server version: {app.version}")
print("Example 1 completed successfully")

## Example 2: jupyter_server functionality test 2

In [ ]:
from jupyter_server.extension.application import ExtensionApp
from jupyter_server.extension.manager import ExtensionPackage

# Example 2: Extension application setup and configuration
print("=== Example 2: Extension Application Setup ===")


# A custom extension is an ExtensionApp subclass with a name and its own settings/handlers hooks
class TestExtension(ExtensionApp):
    name = "test_extension"
    description = "A test extension for demonstration"

    def initialize_settings(self):
        self.settings["test_extension_ready"] = True

    def initialize_handlers(self):
        self.handlers.append((r"/test_extension/ping", object))


extension = TestExtension()
extension.log_level = "INFO"
print(f"Created extension: {extension.name} ({extension.description})")
print(f"Log level: {extension.log_level}")
assert isinstance(extension, ExtensionApp) and extension.name == "test_extension"

extension.initialize_settings()
extension.initialize_handlers()
assert extension.settings["test_extension_ready"] is True
assert [route for route, _ in extension.handlers] == [r"/test_extension/ping"], extension.handlers
print("\u2713 Extension settings and handlers initialized")

# Load a real extension through the loader the server runs at startup. jupyter_server_terminals is a
# dependency of jupyter_server, so it is installed wherever jupyter_server is.
pkg = ExtensionPackage(name="jupyter_server_terminals", enabled=True)
assert pkg.extension_points, "jupyter_server_terminals declares no extension points"
assert pkg.validate(), f"jupyter_server_terminals' extension points do not validate: {pkg.extension_points}"
apps = [point.app for point in pkg.extension_points.values()]
assert any(isinstance(app, ExtensionApp) for app in apps), f"no extension point loads an ExtensionApp: {apps}"
print(f"\u2713 Loaded extension package {pkg.name}: {[type(app).__name__ for app in apps]}")

print("Example 2 completed successfully")

## Example 3: jupyter_server functionality test 3

In [ ]:
from jupyter_server.auth.authorizer import AllowAllAuthorizer, Authorizer
from jupyter_server.auth.identity import User

# Example 3: Authentication and authorization setup
print("=== Example 3: Authentication and Authorization ===")

# Create an Authorizer instance
authorizer = Authorizer()
print(f"Created Authorizer: {type(authorizer).__name__}")

# Create a test user identity
user = User(username="test_user", name="Test User")
print(f"Created user: {user.username} ({user.name})")

# Test authorization for different actions
resource = "contents"
actions = ["read", "write", "execute"]

# The base Authorizer is abstract: is_authorized must refuse to answer rather than allow anything.
base_refused = False
try:
    authorizer.is_authorized(None, user, "read", resource)
except NotImplementedError:
    base_refused = True
assert base_refused, "the base Authorizer.is_authorized answered instead of raising NotImplementedError"
print("✓ The base Authorizer refuses to decide (NotImplementedError)")

# AllowAllAuthorizer is the server's default: it must allow every action. `handler` is unused by it.
allow_all = AllowAllAuthorizer()
print("Testing authorization:")
for action in actions:
    allowed = allow_all.is_authorized(None, user, action, resource)
    assert allowed is True, f"AllowAllAuthorizer refused {action} on {resource}: {allowed!r}"
    print(f"  - {action} on {resource}: allowed")

# Show authorizer configuration
print(f"✓ Authorizer configured successfully")
print(f"✓ User identity system working")

# Test authorizer is ready for server use
if hasattr(authorizer, 'is_authorized'):
    print("✓ Authorization methods available")

print("Example 3 completed successfully")

## Example 4: jupyter_server functionality test 4

In [ ]:
from jupyter_server.services.contents.filemanager import FileContentsManager
import tempfile
import os

# Example 4: Contents manager operations
print("=== Example 4: Contents Manager Operations ===")

# Create temporary directory for testing
with tempfile.TemporaryDirectory() as temp_dir:
    # Create a FileContentsManager (the server's default ContentsManager) rooted in the temp dir
    cm = FileContentsManager(root_dir=temp_dir)
    print(f"Created ContentsManager: {type(cm).__name__}")
    assert cm.root_dir == temp_dir, cm.root_dir

    # Create, save and read back a file through the manager's own API, never through `open`
    model = cm.new_untitled(type="file", ext=".txt")
    assert model["type"] == "file", model
    assert model["name"].endswith(".txt") and model["path"] == model["name"], model
    print(f"✓ Created untitled file: {model['name']}")

    text = "Hello, Jupyter Server!"
    cm.save({"type": "file", "format": "text", "content": text}, model["path"])
    fetched = cm.get(model["path"])
    assert fetched["content"] == text, fetched
    assert fetched["type"] == "file" and fetched["name"] == model["name"], fetched
    assert os.path.isfile(os.path.join(temp_dir, model["path"])), os.listdir(temp_dir)
    print(f"✓ Saved and read back {len(text)} characters through the ContentsManager")

print("Example 4 completed successfully")

## Example 5: jupyter_server functionality test 5

In [ ]:
import jupyter_server
from jupyter_server.base.call_context import CallContext

# Example 5: Server constants and call context usage
print("=== Example 5: Server Constants and Call Context ===")

# Test server constants
default_port = jupyter_server.DEFAULT_JUPYTER_SERVER_PORT
print(f"Default Jupyter server port: {default_port}")

# Test static files path
static_path = jupyter_server.DEFAULT_STATIC_FILES_PATH
print(f"Default static files path: {static_path}")

# Test events schema path
events_schema = jupyter_server.DEFAULT_EVENTS_SCHEMA_PATH
print(f"Events schema path: {events_schema}")

# Demonstrate call context functionality
print("Testing call context:")
try:
    # `call_context` is a module; its CallContext class is a namespace of classmethods over one
    # ContextVar (set/get named values), not a context manager. Run in a copied context so the
    # values stay scoped to it, as they are per request inside the server.
    import contextvars

    def _in_call_context():
        CallContext.set('operation', 'test')
        CallContext.set('user', 'demo')
        print("✓ Call context established")
        return {name: CallContext.get(name) for name in CallContext.context_variable_names()}

    current_context = contextvars.copy_context().run(_in_call_context)
    print(f"Current context: {current_context}")
    assert current_context == {'operation': 'test', 'user': 'demo'}, current_context
    assert CallContext.get('operation') is None, "the value leaked out of the copied context"
    print("✓ Context tracking working")
        
except Exception as e:
    print(f"Call context error: {e}")
    raise

# Show server constants are properly configured
print(f"✓ Server constants loaded successfully")
print(f"Jupyter Server version: {jupyter_server.__version__}")

print("Example 5 completed successfully")